# FinanceFlow Data Profiling & Audit
## Part A — Credit Risk Audit Findings


In [1]:
import pandas as pd
from ydata_profiling import ProfileReport

df = pd.read_csv("financeflow_raw.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Dataset shape: (400, 13)
Columns: ['loan_id', 'business_age_years', 'sector', 'num_employees', 'annual_revenue_usd', 'loan_amount_usd', 'loan_term_months', 'interest_rate_pct', 'credit_score', 'debt_to_income_ratio', 'loan_purpose', 'state', 'default_flag']


C:\Users\gitam\AppData\Local\Temp\ipykernel_3176\1079843176.py:2: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
profile = ProfileReport(df, title="FinanceFlow Loan Data Audit", explorative=True)
profile.to_file("financeflow_profile.html")
print("Profile report saved to financeflow_profile.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:00<00:00, 393.75it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile report saved to financeflow_profile.html


In [3]:
# --- Audit Analysis ---
print("=== Missing Credit Score ===")
missing_cs = df['credit_score'].isna().sum()
print(f"Rows missing credit_score: {missing_cs}")

print("\n=== Negative Annual Revenue ===")
neg_rev = (df['annual_revenue_usd'] < 0).sum()
print(f"Rows with negative annual_revenue_usd: {neg_rev}")

print("\n=== Missing Default Flag ===")
missing_df = df['default_flag'].isna().sum()
print(f"Rows missing default_flag: {missing_df}")

print("\n=== Default Rate in Clean Rows ===")
clean = df.dropna(subset=['default_flag'])
default_rate = clean['default_flag'].mean()
print(f"Default rate in clean rows: {default_rate:.1%}")
print(f"Industry benchmark: 9.8%")


=== Missing Credit Score ===
Rows missing credit_score: 30

=== Negative Annual Revenue ===
Rows with negative annual_revenue_usd: 15

=== Missing Default Flag ===
Rows missing default_flag: 25

=== Default Rate in Clean Rows ===
Default rate in clean rows: 28.5%
Industry benchmark: 9.8%


## Credit Risk Audit Findings

**How many loan applications are missing credit_score?**
30 rows are missing credit_score. Credit score is a blocking field because it is the single most predictive input for consumer credit risk — without it, the model cannot distinguish high-risk from low-risk applicants. Regulators (e.g. ECOA) also require this input for demonstrably fair lending decisions.

**How many rows have negative annual_revenue_usd?**
15 rows. Two plausible explanations: (1) data entry error — a negative sign was accidentally included when recording revenue; (2) the business actually had negative revenue (i.e., net losses). The second is more dangerous for a lender because it suggests the business cannot service debt, yet the raw data would look like a small negative number that a model might misinterpret as a data error rather than a distress signal.

**How many rows are missing default_flag?**
25 rows. These cannot be used in model training because default_flag is the target variable — without knowing whether a loan defaulted, the model has nothing to learn from.

**What is the default rate in clean rows?**
Calculated above. Compare to the industry benchmark of 9.8%.

**One sentence for the CFO:**
The single issue that most delays the AI project is the 30 missing credit scores — since credit score is the most important predictor of default, we cannot train a reliable model until these are recovered from the core banking system or supplemented from a bureau feed.
